# 変換候補を複数出す (N-best)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/31_japanese_input/kkc_nbest.ipynb)

NLP クックブック 31-06 ／ 日本語入力（かな漢字変換）

第 1 候補が間違っていたときに、利用者が選び直せるよう、変換候補を確からしい順に複数出したい。

In [1]:
!pip install -q fugashi unidic-lite jaconv marisa-trie

In [2]:
import heapq, io, math, re, urllib.request, zipfile
from collections import Counter, defaultdict
import jaconv
import marisa_trie
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/000148/files/"
dic, unigram, bigram = defaultdict(set), Counter(), Counter()
for f in ["789_ruby_5639.zip", "752_ruby_2438.zip", "773_ruby_5968.zip"]:
    for s in re.split(r"(?<=。)|\n", aozora(base + f)):
        if not s.strip():
            continue
        prev = "<s>"; unigram[prev] += 1
        for w in tagger(s.strip()):
            surface, reading = w.surface, jaconv.kata2hira(w.feature.kana or w.surface)
            dic[reading].add(surface); unigram[surface] += 1; bigram[prev, surface] += 1; prev = surface
        bigram[prev, "</s>"] += 1
total, trie = sum(unigram.values()), marisa_trie.Trie(dic.keys())

def cost(prev, word, lam=0.8):
    p = lam * (bigram.get((prev, word), 0) / unigram[prev] if unigram.get(prev) else 0.0) + (1 - lam) * unigram.get(word, 0) / total
    return -math.log(p) if p > 0 else 20.0

In [3]:
def nbest(kana, n=5, beam=30):
    """ビームサーチ: 各位置で、コストの低い順に beam 個の途中結果 (コスト, 単語列) を残しながら進む"""
    beams = [[] for _ in range(len(kana) + 1)]
    beams[0] = [(0.0, ("<s>",))]
    for i in range(len(kana)):
        beams[i] = heapq.nsmallest(beam, beams[i])
        candidates = [(i + len(r), s) for r in trie.prefixes(kana[i:]) for s in dic[r]] + [(i + 1, kana[i])]
        for c, words in beams[i]:
            for j, word in candidates:
                beams[j].append((c + cost(words[-1], word), words + (word,)))
    final = sorted((c + cost(words[-1], "</s>"), words[1:]) for c, words in beams[-1])
    results, seen = [], set()
    for c, words in final:                       # 区切り方だけが違って、表記が同じものは 1 つにまとめる
        text = "".join(words)
        if text not in seen:
            seen.add(text); results.append((text, round(c, 1), "|".join(words)))
        if len(results) == n:
            break
    return results

for kana in ["かみをきった", "じしんがある", "きかいがない", "はなしをきく"]:
    print(kana)
    for text, c, seg in nbest(kana):
        print(f"   {c:6.1f}  {text}  ({seg})")

かみをきった
     38.5  髪を切った  (髪|を|切っ|た)
     41.5  紙を切った  (紙|を|切っ|た)
     41.9  神を切った  (神|を|切っ|た)
     42.6  かみを切った  (かみ|を|切っ|た)
     45.0  髪をきった  (髪|を|きっ|た)
じしんがある
     35.2  自身がある  (自身|が|ある)
     36.8  自信がある  (自信|が|ある)
     37.9  地震がある  (地震|が|ある)
     39.5  時心がある  (時|心|が|ある)
     40.6  自身が有る  (自身|が|有る)
きかいがない
     36.1  機会がない  (機会|が|ない)
     37.4  器械がない  (器械|が|ない)
     41.4  機会が無い  (機会|が|無い)
     42.0  機会が鳴い  (機会|が|鳴い)
     42.6  器械が無い  (器械|が|無い)
はなしをきく
     35.3  話を聞く  (話|を|聞く)
     36.6  話をきく  (話|を|きく)
     37.2  話を聴く  (話|を|聴く)
     37.4  話しを聞く  (話し|を|聞く)
     38.2  話を利く  (話|を|利く)


In [4]:
# 文節ごとに候補を選び直す操作のために: 文の一部分だけについて候補を出す
def candidates_for(kana, left="<s>", n=6):
    """読み全体が 1 語になる候補を、直前の単語 left との並びやすさの順に返す"""
    return sorted(dic.get(kana, {kana}), key=lambda s: cost(left, s))[:n]

print("「が」の後ろの「きく」:", candidates_for("きく", left="が"))
print("「を」の後ろの「きく」:", candidates_for("きく", left="を"))
print("文頭の「かみ」:", candidates_for("かみ"))
print("「の」の後ろの「かみ」:", candidates_for("かみ", left="の"))

「が」の後ろの「きく」: ['聞く', 'きく', '利く', '聴く', '菊']
「を」の後ろの「きく」: ['聞く', 'きく', '聴く', '利く', '菊']
文頭の「かみ」: ['神', 'かみ', '髪', '紙', '噛み', '加味']
「の」の後ろの「かみ」: ['神', '髪', 'かみ', '紙', '噛み', '加味']


- 最良の 1 つだけを求めるビタビアルゴリズムと違い、途中結果を複数残しながら進めると、上位 N 個の変換結果が得られる。厳密な N-best には、後ろ向きの A* 探索を組み合わせる方法が使われる。
- 表記が同じで区切り方だけが違う候補は、利用者から見れば同じなので、まとめて 1 つにする。
- 実際の入力システムでは、文全体の候補を並べるより、文節ごとに候補を出して選び直せるようにするほうが使いやすい。選び直した結果は、次の文節の候補の順序にも影響する。
- 候補の一覧には、ひらがな・カタカナのままの表記も必ず入れておく。辞書にない語を入力する手段がなくなってしまう。